# 4 Composition: a representation you designed, next to one the model learned

> **Optional / appendix, and still on the old dataset.** Not part of the graded sequence (01-08), and not yet ported to the sunspot/solar/electricity roster the rest of this lesson uses: its `ts_features.py` band boundaries (`p<6h`, `p6-10h`, `p10-16h`, `p16-36h`) and ACF lags are hardcoded to an hourly window, not just pointing at this notebook's old data, so retargeting it means redesigning the feature set for daily resolution and re-verifying every numeric claim below, not a loader swap. Its §4.6 routing experiment already moved to `05_gates_highways_moe.ipynb`, which cites the Gate/Highway/MoE sources properly and runs on the current dataset. The *rest* of this notebook -- hand-crafted features vs. a learned representation, where two streams should fuse, and the three ablations that catch a false positive -- is real, measured, and worth keeping findable even on its original data; retargeting it is tracked as follow-up work, not done here.

---


So far every model in this lesson has been handed the same thing: 96 raw numbers. This
notebook gives it a **second** description of the same window — thirty features that a
person wrote down on purpose, each with a name and a reason: how much energy sits at
daily periods, how steep the trend is, what the autocorrelation at lag 24 looks like,
what hour of the week it is, what the six load channels were doing.

That is the pattern `4_attention/06_composition` runs on text, where the designed stream
is sixty regexes over legal language. Same idea, different modality, and the same tool
does the joining: **vectormesh**'s composition components (`Parallel`, `Serial`,
`Concatenate2D`, `Concatenate3D`, `Gate`, `Highway`, `MoE`).

Three questions, in order:

1. How far do thirty hand-made numbers get you *on their own*?
2. *Where* should two streams meet — after each is squashed to a vector, or while the
   time axis still exists?
3. Does any of it help?

Question 3 has a measured answer and it is **no** — and getting to a trustworthy *no*
takes three ablations, one of which overturns the answer to question 2 entirely. That
sequence is the notebook. A clean win would have taught you less.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings

from ett import COVARIATE_NAMES, load_ett, window_index
from ts_features import patch_features, window_features

W, H, PATCH = 96, 24, 24        # 96-hour window = 4 patches of 24 hours
D, EPOCHS, SEEDS = 32, 10, (0, 1, 2)

ett = load_ett()
mu, sigma = ett.values[ett.splits["train"]].mean(), ett.values[ett.splits["train"]].std()


def build(split: str):
    sl = ett.splits[split]
    values = (ett.values[sl] - mu) / sigma
    idx = window_index(len(values), W + H, 1)
    win = values[idx]
    x, y = win[:, :W], win[:, W:]
    origins = ett.timestamps[sl][idx[:, W - 1].numpy()]     # timestamp of the forecast origin
    covariates = ett.covariates[sl][idx[:, :W]]
    features, names = window_features(x, origins, covariates, COVARIATE_NAMES)
    patched_features, patch_names = patch_features(x, PATCH)
    return x, y, features, patched_features, names, patch_names


xtr, ytr, ftr, pftr, NAMES, PATCH_NAMES = build("train")
xva, yva, fva, pfva, _, _ = build("val")
xte, yte, fte, pfte, _, _ = build("test")

# designed features come in wildly different units; standardise them on train statistics too
fmu, fsd = ftr.mean(0), ftr.std(0).clamp(min=1e-6)
ftr, fva, fte = (ftr - fmu) / fsd, (fva - fmu) / fsd, (fte - fmu) / fsd
pmu = pftr.reshape(-1, pftr.shape[-1]).mean(0)
psd = pftr.reshape(-1, pftr.shape[-1]).std(0).clamp(min=1e-6)
pftr, pfva, pfte = (pftr - pmu) / psd, (pfva - pmu) / psd, (pfte - pmu) / psd

F, FP, C = ftr.shape[1], pftr.shape[2], W // PATCH
print(f"raw window      {tuple(xtr.shape)}")
print(f"designed vector {tuple(ftr.shape)}       {F} features per window")
print(f"designed matrix {tuple(pftr.shape)}   {FP} features per patch, {C} patches")

## 4.1 The features, and why each one is there

Read `ts_features.py` before running further — it is short, and every feature in it is
an argument. A few of them, and the claim each makes:

| feature | the claim |
|---|---|
| `band:p16-36h` | *the daily cycle carries this fraction of the window's energy* |
| `spectral_entropy` | *is this structured or is it noise?* |
| `trend_slope`, `trend_r2` | *how fast is it moving, and is that a trend or one spike?* |
| `acf_24` | *does this hour look like the same hour yesterday?* |
| `hour_sin`, `dow_cos`, `is_weekend` | *where in the day and week is the forecast origin?* |
| `cov_last:HUFL` … | *what were the six load channels doing at the origin?* |

The calendar ones deserve a note: **they are the only features here that are not
functions of the window at all.** No amount of staring at 96 temperature readings tells
you it is a Tuesday. If designed features are going to add anything, that is where you
would bet on it — hold the thought until section 4.7, where it is tested.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for a, name in zip(ax, ("band:p16-36h", "trend_slope", "acf_24")):
    i = NAMES.index(name)
    a.plot(ftr[:1500, i], lw=0.6)
    a.set_title(name, fontsize=9)
    a.set_xlabel("window index (train)")
plt.tight_layout()

print(f"{'feature':22s} {'mean':>8} {'std':>8}   (after standardising on train)")
for name in ("band:p16-36h", "spectral_entropy", "trend_slope", "acf_24", "is_weekend", "cov_last:HUFL"):
    i = NAMES.index(name)
    print(f"{name:22s} {ftr[:, i].mean():8.3f} {ftr[:, i].std():8.3f}")
print(f"\nper-patch features: {PATCH_NAMES}")

Note the price tag. Computing all 30 features for all 8500 windows took a fraction of a
second, and no GPU was involved. A designed representation is **expensive in your time
and nearly free in compute**; a learned one is exactly the other way round. That
trade-off is the designed/learned line from lesson 1, now with numbers on both sides.

## 4.2 How far do thirty numbers get you on their own?

Before fusing anything, run each stream by itself. Same trainer, same seeds, same
everything — only the input changes.

In [ ]:
def collate(batch):
    # two input tensors and a target; the model receives a *tuple*, which is what
    # vectormesh's Parallel and Concatenate components consume
    x1 = torch.stack([b[0] for b in batch])
    x2 = torch.stack([b[1] for b in batch])
    y = torch.stack([b[2] for b in batch])
    return (x1, x2), y


def mae(y, yhat):
    return float((y - yhat).abs().mean())


results: dict[str, dict] = {}


def fit(make_model, train_set, val, test, name: str, seeds=SEEDS, epochs=EPOCHS):
    # vectormesh.training.Trainer handles a tuple input (x1, x2) natively -- see
    # Trainer._to_device -- so the tuple collate function above drops straight in.
    # log_dir is never created on disk unless something writes to it, and progress=False
    # drops the tqdm bars, keeping one Trainer per seed cheap across the many fit() calls
    # below.
    val_scores, test_scores, secs = [], [], []
    validloader = DataLoader(TensorDataset(val[0], val[1], val[2]), batch_size=len(val[0]),
                             collate_fn=collate)
    for seed in seeds:
        torch.manual_seed(seed)
        model = make_model()
        trainloader = DataLoader(train_set, batch_size=64, shuffle=True, collate_fn=collate)
        settings = TrainerSettings(
            epochs=epochs,
            metrics=[MAE()],
            logdir=Path("modellogs").resolve(),
            train_steps=len(trainloader),
            valid_steps=len(validloader),
            optimizer_kwargs={"lr": 1e-3},
            earlystop_kwargs=None,
        )
        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=nn.MSELoss(),
            optimizer=torch.optim.Adam,
            traindataloader=trainloader,
            validdataloader=validloader,
            progress=False,
        )
        t0 = time.time()
        result = trainer.loop()
        secs.append(time.time() - t0)
        val_scores.append(result.metric_dict["MAE"])
        with torch.no_grad():
            test_scores.append(mae(test[2], model((test[0], test[1]))))
    results[name] = {"mae": float(np.mean(val_scores)), "std": float(np.std(val_scores)),
                     "test": float(np.mean(test_scores)),
                     "params": sum(p.numel() for p in model.parameters()),
                     "secs": float(np.mean(secs))}
    r = results[name]
    print(f"{name:38s} val {r['mae']:.4f} +/- {r['std']:.4f}   test {r['test']:.4f}   "
          f"{r['params']:>7,}p   {r['secs']:5.1f}s")


flat_train = TensorDataset(xtr, ftr, ytr)
flat_val, flat_test = (xva, fva, yva), (xte, fte, yte)


class LinearOn(nn.Module):
    def __init__(self, which: int, size: int):
        super().__init__()
        self.which, self.fc = which, nn.Linear(size, H)

    def forward(self, X):
        return self.fc(X[self.which])


fit(lambda: LinearOn(0, W), flat_train, flat_val, flat_test, "linear on the raw window (96)")
fit(lambda: LinearOn(1, F), flat_train, flat_val, flat_test, "linear on 30 designed features")

Thirty numbers, **744 parameters**, and it lands within a few percent of the model that
sees all 96 raw values. That is a real result and it is the case *for* designing
features: a third of the parameters, an input you can read, and a model whose weights
you could argue about with a domain expert.

It is also, as the next four sections establish, not the same thing as those features
being *useful* once the raw window is already on the table.

## 4.3 The simplest possible fusion

Before reaching for a library: glue the two vectors together and run one linear layer
over the 126 numbers. If the designed features carry information the raw window does
not, this must beat both rows above.

In [ ]:
class Concatenated(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(W + F, H)

    def forward(self, X):
        return self.fc(torch.cat(X, dim=1))


fit(Concatenated, flat_train, flat_val, flat_test, "linear on [raw ++ features] (126)")

On validation it edges the raw-only model by an amount inside the seed noise. On test it
is clearly **worse**. Six hundred extra parameters, no gain, and slightly more to overfit
with — the first hint of where this notebook ends up, at the cheapest possible price.

Keep going anyway, because "we fused it wrong" is a real possibility and the whole point
of the composition machinery is that it makes the alternatives cheap to try.

## 4.4 Fusion altitude: where should the streams meet?

`06_composition` in lesson 5 poses the question this way, and it is the right question:

- **After aggregation.** Each stream is reduced to one vector per window, then the
  vectors are glued. `Parallel` runs the branches, `Concatenate2D` joins them. By the
  time they meet, the time axis is gone.
- **Per chunk.** Cut the window into 4 patches of 24 hours; each patch has a raw slice
  *and* its own designed features. `Concatenate3D` joins them while the time axis still
  exists, so downstream components see each patch's numbers next to that same patch's
  features.

The second only means anything if patch *i* of one stream really does describe patch *i*
of the other. That is the **alignment contract**, and nothing in PyTorch enforces it.

In [ ]:
def patchify(x):
    return x.reshape(len(x), C, PATCH)          # (B, 96) -> (B, 4, 24)


chunk_train = TensorDataset(patchify(xtr), pftr, ytr)
chunk_val, chunk_test = (patchify(xva), pfva, yva), (patchify(xte), pfte, yte)

print(f"raw patches      {tuple(patchify(xtr).shape)}")
print(f"patch features   {tuple(pftr.shape)}")
print(f"same patch count: {patchify(xtr).shape[1] == pftr.shape[1]}")

# both were cut with PATCH=24 from the same window, so row i means the same thing in both.
# check it explicitly rather than believing the comment:
manual_mean_of_patch_2 = xtr[:5, 2 * PATCH:3 * PATCH].mean(1)
feature_mean_of_patch_2 = pftr[:5, 2, PATCH_NAMES.index("mean")] * psd[PATCH_NAMES.index("mean")] + pmu[PATCH_NAMES.index("mean")]
print(f"\npatch 2's mean, computed two ways: {torch.allclose(manual_mean_of_patch_2, feature_mean_of_patch_2, atol=1e-5)}")

### The bug that does not crash

Now break the contract the way somebody actually would — by reading a patch length off a
colleague's config instead of off the data you are fusing with.

In [ ]:
from vectormesh.components import Concatenate3D

wrong, _ = patch_features(xtr[:8, -64:], patch_len=16)      # a different window AND a different patch size
right = patchify(xtr[:8])

print(f"raw patches     {tuple(right.shape)}")
print(f"wrong features  {tuple(wrong.shape)}   <- same number of patches, by coincidence")
fused = Concatenate3D()((right, ((wrong - pmu) / psd)))
print(f"Concatenate3D   {tuple(fused.shape)}   no error, no warning")
print(f"\npatch 0 of the raw stream covers hours   0-{PATCH}")
print(f"patch 0 of the wrong feature stream covers hours {W - 64}-{W - 64 + 16}")

It runs clean and it is meaningless. A 16-step patch over the last 64 hours happens to
produce four rows, so every shape matches and `torch.cat` is satisfied — while row 0 of
one stream describes a completely different stretch of time than row 0 of the other.

This is the sharpest form of the formal/semantic split in `tensor-ladder.md`: **jaxtyping
catches rank and dimension errors, and nothing mechanical catches semantic
misalignment.** The defence is to derive both streams from one source of truth (here,
one `PATCH` constant used twice) and then verify, as the previous cell did.

### Both altitudes, measured

In [ ]:
from vectormesh.components import (Concatenate2D, MeanAggregator, NeuralNet,
                                   Parallel, Projection, Serial)


def doc_fusion():
    # each branch reduces its own stream to (B, D); Concatenate2D glues the two vectors
    return Serial([
        Parallel([
            Projection(hidden_size=W, out_size=D),      # (B, 96) -> (B, 32)
            Projection(hidden_size=F, out_size=D),      # (B, 30) -> (B, 32)
        ]),
        Concatenate2D(),                                # -> (B, 64)
        NeuralNet(hidden_size=2 * D, out_size=H),
    ])


def chunk_fusion_mean():
    # join while the time axis exists, then average the patches away
    return Serial([
        Concatenate3D(),                                       # (B,4,24)+(B,4,11) -> (B,4,35)
        Projection(hidden_size=PATCH + FP, out_size=D),        # -> (B, 4, 32)
        MeanAggregator(),                                      # -> (B, 32)
        NeuralNet(hidden_size=D, out_size=H),
    ])


fit(doc_fusion, flat_train, flat_val, flat_test, "doc fusion (Parallel -> Concatenate2D)")
fit(chunk_fusion_mean, chunk_train, chunk_val, chunk_test, "chunk fusion (Concatenate3D -> mean)")

Both are **worse than a single linear layer on the raw window**, and chunk fusion — the
sophisticated one, the one the alignment contract was for — is the worst row in the
notebook so far.

At this point the tempting conclusion is "fusion does not work here". Do not take it.
Before believing a comparison, check that the thing you varied is the thing that moved.

## 4.5 The diagnosis: it was never the fusion

Look at what `chunk_fusion_mean` does at the end: `MeanAggregator` averages the four
patches into one vector. Averaging is permutation-invariant — it throws away **which
patch was last**. On a series whose `acf(1)` is 0.99, the most recent patch is nearly the
entire answer, and we just averaged it into the other three.

Change exactly one component. Keep the fusion, keep the projection, keep the parameter
budget roughly; replace the mean with a flatten, which keeps position.

In [ ]:
class ChunkFusion(nn.Module):
    # Concatenate3D -> Projection -> [optional per-patch component] -> flatten -> head
    def __init__(self, use_features: bool = True, per_patch=None, per_window=None):
        super().__init__()
        self.use_features = use_features
        self.cat = Concatenate3D()
        self.proj = Projection(hidden_size=PATCH + FP if use_features else PATCH, out_size=D)
        self.per_patch = per_patch() if per_patch else None
        self.per_window = per_window() if per_window else None
        self.head = nn.Linear(C * D, H)

    def forward(self, X):
        z = self.proj(self.cat((X[0], X[1])) if self.use_features else X[0])
        if self.per_patch is not None:
            z = self.per_patch(z)
        z = z.flatten(1)                      # (B, 4, 32) -> (B, 128): position survives
        if self.per_window is not None:
            z = self.per_window(z)
        return self.head(z)


fit(ChunkFusion, chunk_train, chunk_val, chunk_test, "chunk fusion (Concatenate3D -> flatten)")

One component swapped, and it is now the **best row in the notebook** — better than the
linear baseline on both splits, at under twice the parameters.

Sit with what just happened. Section 4.4 compared two fusion altitudes and one lost
badly. That comparison was worthless: the two pipelines differed in their *aggregator*
as well as their fusion point, and the aggregator was doing all the work. **The variable
under study was not the variable that moved the number** — a confound, in a two-row
table, in a notebook explicitly about being careful.

This is why composition libraries earn their keep. `MeanAggregator` → flatten was a
one-line change inside an otherwise identical `Serial`, so the confound was cheap to
find. In a hand-rolled `forward` with the pooling baked in, it is a rewrite, and rewrites
do not get done on a hunch.

## 4.6 The routing family: Gate, Highway, TransformerBlock, MoE

vectormesh's other components slot into the same pipeline. Each is one idea at
increasing resolution:

| component | what it learns | where it fits |
|---|---|---|
| `Gate` | a per-dimension scale, `sigmoid(Wx) * x` | on `(B, dim)` |
| `Highway` | a learned interpolation between `x` and `T(x)` | on `(B, dim)` |
| `TransformerBlock` | lets each patch attend to the others | on `(B, patches, dim)` |
| `MoE` | a softmax-weighted blend of N experts | either |

The mixture-of-experts routes **per patch**: each patch's fused representation gets its
own softmax over two `TransformerBlock` experts. That is (a very small version of) the
architecture family behind most current frontier language models, assembled from parts
you have already met.

In [ ]:
from vectormesh.components import Gate, Highway, MoE, TransformerBlock

fit(lambda: ChunkFusion(per_window=lambda: Gate(hidden_size=C * D)),
    chunk_train, chunk_val, chunk_test, "  + Gate")
fit(lambda: ChunkFusion(per_window=lambda: Highway(
        transform=NeuralNet(hidden_size=C * D, out_size=C * D), hidden_size=C * D)),
    chunk_train, chunk_val, chunk_test, "  + Highway")
fit(lambda: ChunkFusion(per_patch=lambda: TransformerBlock(hidden_size=D, num_heads=2)),
    chunk_train, chunk_val, chunk_test, "  + TransformerBlock (per patch)")
fit(lambda: ChunkFusion(per_patch=lambda: MoE(
        experts=[TransformerBlock(hidden_size=D, num_heads=2) for _ in range(2)],
        hidden_size=D, out_size=D)),
    chunk_train, chunk_val, chunk_test, "  + MoE of 2 TransformerBlocks")

Every one of them makes it worse, and the largest of them is the worst by a distance.
There is nothing mysterious here: each component is a **bet that a particular kind of
structure exists in the data**, and a bet you did not need is just parameters you have to
fit. Four patches of a smooth univariate series do not have much for attention to attend
to, and no amount of routing invents structure that is not there.

That is the same argument section 3.10 made about Autoformer, arrived at from a different
direction. Capacity is not free, and "state of the art" is a statement about a regime.

## 4.7 The ablation that keeps you honest

The best row is `chunk fusion (Concatenate3D -> flatten)`. Before writing that up as
*designing features helped*, do the one-variable ablations that could falsify it. Two of
them, both cheap:

1. Replace the designed features with **Gaussian noise** of the same shape. If the score
   holds up, the features were never being used.
2. Delete the second stream entirely and keep the pipeline otherwise identical.

In [ ]:
noise_train = TensorDataset(patchify(xtr), torch.randn_like(pftr), ytr)
noise_val = (patchify(xva), torch.randn_like(pfva), yva)
noise_test = (patchify(xte), torch.randn_like(pfte), yte)

fit(ChunkFusion, noise_train, noise_val, noise_test, "  same pipeline, features -> noise")
fit(lambda: ChunkFusion(use_features=False), chunk_train, chunk_val, chunk_test,
    "  same pipeline, no feature stream")

There it is. The pipeline with **no designed features at all** scores the same as the one
with them — the two differ by less than a fifth of the seed spread, on both splits — and
swapping the features for pure noise costs barely more. The model was not using the
second stream. The win in 4.5 came entirely
from **patching the raw window and keeping position**, which is to say: from PatchTST's
first idea, which you met in `03_probabilistic`.

Had we stopped at 4.5 — a new best score, a satisfying story about fusing designed
features with learned ones — the notebook would have shipped a conclusion that is
straightforwardly false. It cost two `fit` calls to find out.

## 4.8 Why the features had nothing to add

"The model ignored them" invites the question *why*. There is an answer, and it is
checkable: **a linear layer on the raw window can already compute most of these features
by itself.** `mean` is one particular weighted sum of the 96 inputs. So is `last`. So is
`trend_slope`. Handing a linear model a linear function of its own input adds a column
and no information.

Regress each feature on the raw window and look at how much of it is recoverable.

In [ ]:
A = torch.cat([xtr, torch.ones(len(xtr), 1)], dim=1)
coef = torch.linalg.lstsq(A, ftr).solution
residual = ftr - A @ coef
r2 = 1 - (residual**2).sum(0) / ((ftr - ftr.mean(0)) ** 2).sum(0)

print("how much of each designed feature is a linear function of the raw window?\n")
print(f"{'feature':22s} {'R^2':>7}")
for name, value in sorted(zip(NAMES, r2.tolist()), key=lambda t: -t[1]):
    marker = "  <- already in the span" if value > 0.9 else ""
    print(f"{name:22s} {value:7.3f}{marker}")

`trend_slope`, `mean`, `last` and `last_minus_mean` come back with **R² = 1.000**: they
are exactly linear functionals of the window, and a linear layer has them for free. `min`
and `max` are nearly so. Even `hour_sin` is two-thirds recoverable — the oil temperature
quietly encodes the time of day, which is a nice thing to learn about the data.

The bottom of the list is where the genuinely new information is: the spectral bands
(quadratic in the input, so not in a linear layer's reach), the autocorrelations, the
day-of-week features, and the six load channels — which are not derived from the window
at all.

So run the sharpest version of the test. Fuse **only** the features a linear layer
demonstrably cannot compute for itself.

In [ ]:
outside = [i for i, v in enumerate(r2.tolist()) if v < 0.9]
print(f"{len(outside)} of {F} features have R^2 < 0.9")

sub_train = TensorDataset(xtr, ftr[:, outside], ytr)
sub_val, sub_test = (xva, fva[:, outside], yva), (xte, fte[:, outside], yte)


class ConcatenatedSubset(nn.Module):
    def __init__(self, k: int):
        super().__init__()
        self.fc = nn.Linear(W + k, H)

    def forward(self, X):
        return self.fc(torch.cat(X, dim=1))


fit(lambda: ConcatenatedSubset(len(outside)), sub_train, sub_val, sub_test,
    f"linear on [raw ++ {len(outside)} span-outside features]")

Still nothing — and worse on test than raw alone. The features are not redundant *and*
they still do not help. The remaining explanation is the ordinary one: at a 24-hour
horizon on a thermal series, almost all the predictable signal is in the recent values,
and everything else is competing for the same small residual with extra parameters.

## 4.9 The table

Every row, sorted by the split we did not tune on.

In [ ]:
print(f"{'model':42s} {'val':>8} {'+/-':>7} {'test':>8} {'params':>9} {'secs':>6}")
print("-" * 84)
for name in sorted(results, key=lambda k: results[k]["test"]):
    r = results[name]
    print(f"{name:42s} {r['mae']:8.4f} {r['std']:7.4f} {r['test']:8.4f} {r['params']:9,} {r['secs']:6.1f}")

## What to take from this

- **A designed representation is cheap in compute and expensive in your time.** Thirty
  features, 744 parameters, and it nearly matched the raw window on its own. That is
  worth knowing even though it did not help in combination.
- **Before fusing, ask what your model can already compute.** The R² table in 4.8 is
  five lines and it would have predicted the whole outcome. A feature that is a linear
  function of an input a linear model already sees is a column of nothing.
- **A designed feature earns its place when it brings information from outside the
  window** — the calendar, an exogenous sensor, a domain fact. That is why the regex
  stream in lesson 5 is a fairer test of the idea than this one: a frozen text embedding
  genuinely cannot count statutory citations, whereas a linear layer genuinely can
  compute a mean.
- **One-variable ablations, always.** Two of them here overturned two conclusions: the
  aggregator confound in 4.5, and the noise control in 4.7. Both cost a single `fit`
  call. A result that has not survived an ablation is a hypothesis.
- **Composition machinery is what makes those ablations cheap.** The value of `Serial`,
  `Parallel` and the rest is not that the pipelines score well — here they mostly did not
  — it is that swapping one component is one line, so you actually run the control
  instead of arguing about it.
- **Every extra component is a bet on structure.** Gate, Highway, attention, MoE all cost
  parameters and all lost. Add capacity when you have identified something for it to do.

## Exercises

**E1 — a series where the calendar matters.** ETTh1's oil temperature is thermal and
smooth. Re-run 4.2 and 4.3 on `Monash-University/monash_tsf` config `pedestrian_counts`,
where footfall is dominated by hour-of-day and weekday-versus-weekend. Do the calendar
features earn their place there? Predict the answer from the spectrum first
(`02_basemodels` section 2.1).

**E2 — features the model cannot fake, part two.** Add features from the *covariates*
that are not just their last value: the load channels' own trend, their correlation with
`OT` over the window, the lag at which they best predict it. Oil temperature is physically
caused by load. Does causal knowledge, encoded by hand, beat a linear layer that sees the
same six numbers?

**E3 — patching is the real finding.** Section 4.5's win came from patch-plus-flatten, not
from fusion. Sweep `PATCH` over 6, 12, 24, 48 with `use_features=False` and find where it
stops helping. Compare with `03_probabilistic`'s PatchTST, which does the same thing with
attention on top.

**E4 — the misalignment, scored.** Train `ChunkFusion` on the deliberately broken feature
stream from 4.4 and add it to the table. Given 4.7, predict its score before you run it —
and say what that prediction implies about how much a silent alignment bug would cost you
on a dataset where the features *did* matter.

**E5 — a fair fight for the designed stream.** Give the raw stream a handicap it cannot
overcome: feed both models only the last 12 hours of the window, but let the designed
features still be computed over all 96. Now the features carry information the model
cannot reach. Do they help? This is the setup under which designed features are actually
deployed.